<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/main/module-10-tuning-and-evaluation/lesson-10.5-finetune-slm/notebooks/GCP_Capstone_10.5_FineTuneSLM.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 10.5 Fine-tune Your Own Small Model — The Same File, a Model You Keep
**Netsetos GenAI Engineering — GCP Capstone** · Module 10 · rebuilt on the live lane, 9 September 2026

Teach a small Gemma the lane's habit - answer, cite by number, quote, say whether it was answerable - on the same frozen dataset 10.1 tuned Gemini on, in its chat format. The decision gate first; the data's provenance read off a real usage row; consent as the tenant's; DLP live through the one list, with a poisoned row to prove the gate can fail; QLoRA on a free T4 behind a GPU check; the GGUF into the datasets bucket under the one name 11.4 builds from; the Modelfile generated from the tokenizer; and a manifest beside the file.

Cells 1 to 5 and 8 to 10 need no GPU. Cells 6 and 7 train and export on a T4 (Runtime → Change runtime type → T4 GPU) and say so when there is none.


## Setup


In [ ]:
!pip install -q google-genai==2.22.0 google-cloud-storage==3.13.1 google-cloud-dlp==3.39.0 requests==2.34.2

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project the lane runs in (make up, lesson 4.8)
REGION     = "us-central1"
TENANT     = "acme"
KIT        = "/content/agentic-ai-weekend-gcp-learners"   # the kit: deploy/shared is the tool layer every lesson on the lane imports
BRANCH     = "main"        # the learner repo's branch: the notebooks and the kit (deploy/) ship there together

import os, subprocess, sys
import google.auth
from google.auth.transport.requests import AuthorizedSession
from google import genai
from google.genai import types

if not os.path.isdir(KIT):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "-b", BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp-learners", KIT], check=True)
sys.path.insert(0, f"{KIT}/deploy")                   # `from shared import ...` - the same layer every service imports

# The lane's URLs are deterministic: service name + project NUMBER (eventarc.tf builds them the same way).
creds, _ = google.auth.default()
NUMBER = AuthorizedSession(creds).get(
    f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
API_URL       = f"https://documind-api-{NUMBER}.{REGION}.run.app"
UPLOAD_BUCKET = f"{PROJECT_ID}-uploads"     # storage.tf: the bucket eventarc.tf watches - the corpus, media included
MEDIA_BUCKET  = f"{PROJECT_ID}-media"       # storage.tf: generated assets, 30-day lifecycle (a cache, not a record)
DATASETS      = f"{PROJECT_ID}-datasets"    # storage.tf (Module 10): the frozen tuning dataset and 10.5's GGUF
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": "global",              # Gemini 3.x generation is served from the global endpoint
    "GOOGLE_GENAI_USE_VERTEXAI": "TRUE",
    "DOCUMIND_PROFILE": "gcp",
    "RAG_API_URL": API_URL,
    "RAG_TIMEOUT_S": "90",                          # 7.2's finding: a cold API takes longer than the default 20 s
    # A notebook has no metadata server to be anyone with: the kit mints its ID tokens AS this roster
    # member (7.1). On Cloud Run the service's own account is the identity and nothing is set.
    "DOCUMIND_IMPERSONATE_SA": f"documind-ui-sa@{PROJECT_ID}.iam.gserviceaccount.com",
})
MEMBER_SA   = os.environ["DOCUMIND_IMPERSONATE_SA"]
OUTSIDER_SA = f"documind-outsider-sa@{PROJECT_ID}.iam.gserviceaccount.com"   # IAM admits it, no roster does (4.8, 7.2)

from shared import documind_tools    # THE one retrieve(). Imported, never pasted - the contract gate fails a paste.
gen = genai.Client(enterprise=True, project=PROJECT_ID, location="global")   # every generate_content in this lesson
APPROVED_FOR_TRAINING = {"acme"}   # tenants who signed for this, by name - the builder takes ONE tenant per file

print("kit:", KIT, "| API:", API_URL, "| datasets:", f"gs://{DATASETS}/sft/")


## Cell 1: The API and its usage rows


In [ ]:
import json, requests, time, subprocess, datetime
from google.cloud import storage

# THE API, CALLED THE WAY THE UI CALLS IT: one ID token per request, minted AS the roster member,
# audience = the API (7.3's hour-long fuse never arms). The kit mints it (documind_tools._id_token).
def api(path: str, body: dict | None = None, base: str | None = None, timeout: int = 120) -> tuple[int, dict | str]:
    """POST one API route (or a candidate revision's, with base=) as documind-ui-sa. Returns (status, json-or-text)."""
    url = (base or API_URL).rstrip("/")
    r = requests.post(f"{url}{path}", json=body,
                      headers={"Authorization": f"Bearer {documind_tools._id_token(url)}"}, timeout=timeout)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]

# The usage rows the API logs - the ONE shape every observability consumer reads (12.3, tenant_daily). On
# the lean lane they live in Cloud Logging; this reads the last few for a surface, newest first.
def usage_rows(minutes: int = 15, limit: int = 20, event: str = "query") -> list[dict]:
    since = (datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(minutes=minutes)).strftime("%Y-%m-%dT%H:%M:%SZ")
    r = subprocess.run(["gcloud", "logging", "read",
                        f'resource.type="cloud_run_revision" AND resource.labels.service_name="documind-api" '
                        f'AND jsonPayload.event="{event}" AND timestamp>="{since}"',
                        "--project", PROJECT_ID, "--limit", str(limit), "--format=json"], capture_output=True, text=True)
    try:
        return [e["jsonPayload"] for e in json.loads(r.stdout or "[]")]
    except ValueError:
        return []

gcs = storage.Client(project=PROJECT_ID)

def gcs_text(uri: str) -> str:
    bucket, _, name = uri.removeprefix("gs://").partition("/")
    return gcs.bucket(bucket).blob(name).download_as_text()

sys.path.insert(0, f"{KIT}/deploy/evals")               # the kit's builders and judges: make_trainset, judge, tune, run_eval
sys.path.insert(0, f"{KIT}/deploy/services/rag-api")    # the API's own modules: cache_manager, router, breakers, cost
print("helpers: api(), usage_rows(), gcs_text(); the kit's evals/ and rag-api/ on sys.path")


## Cell 2: Should you fine-tune at all?
Four habits worth paying for, and they are the only four.


In [ ]:
# First question, before any of this: should you fine-tune at all?
#
# Lesson 10.1 gave the order: prompt, then RAG, then fine-tune. Fine-tuning
# does NOT teach a model new facts - that is what retrieval is for. It teaches
# a model a new HABIT.
#
# Four habits worth paying for, and they are the only four:
def should_finetune(reason: str) -> tuple[bool, str]:
    """Plain-English gate. Say why, out loud, before you spend a GPU hour."""
    good = {
        'format':    'the answer must always look the same - our citation style, every time',
        'latency':   'a small local model answers faster than a call to a big remote one',
        'cost':      'we ask the same narrow kind of question a million times a month',
        'residency': 'the answer must be produced inside India, on our own machine',
    }
    bad = {
        'facts':     'RAG. The model does not need to memorise the HR policy - retrieve it.',
        'freshness': 'RAG. A tuned model is frozen on the day you tuned it.',
        'accuracy':  'usually the prompt or the retrieval. Fix those first and re-measure.',
        'because':   'not a reason.',
    }
    if reason in good:
        return True, good[reason]
    return False, bad.get(reason, 'unknown reason - write it down in one sentence first')


for r in ('format', 'cost', 'residency', 'facts', 'freshness', 'accuracy'):
    ok, why = should_finetune(r)
    print(f'  {r:10} {"TUNE" if ok else "do not":8}  {why}')

print()
print('  DocuMind fine-tunes for FORMAT and RESIDENCY: every answer must carry a')
print('  [Source N] citation with a quote, and one bank customer needs the answer produced')
print('  in India. Neither of those is a fact the model has to memorise.')


## Cell 3: Where the training data is not
A real usage row: no question, no answer, by design. The corpus is the source; the builder is the kit's.


In [ ]:
# WHERE THE TRAINING DATA IS NOT. The obvious answer is "the logs". Read a real usage row from the lane and
# look at what it holds - and at what it does not. The question and the answer are absent, and that is not
# an oversight: sink.tf keeps them out, because a table holding every question every customer ever typed is
# a data-protection problem you built for yourself. The answer cache holds real pairs - and on the lane it
# holds the golden questions every eval run asked: the TEST set. So the training data comes from the corpus,
# through the kit's builder (10.1), and this lesson reads the same frozen file in its chat format.
row = (usage_rows(minutes=60 * 24, limit=1) or [{}])[0]
print("a usage row carries:", sorted(row.keys()))
assert row and "question" not in row and "answer" not in row, "the row is empty (ask a question first) or the sink leaks the text"
print("\nnot in that list: the question, and the answer. The logs cannot build a training set, by design.")
print("the corpus can - make trainset - and the file it froze is at", f"gs://{DATASETS}/sft/documind_sft_v1.chat.jsonl")


## Cell 4: Permission comes before data
One tenant per file; every row through `eligible()`.


In [ ]:
# PERMISSION COMES BEFORE DATA. The corpus is the tenant's; a training set built from it is a NEW purpose,
# and "we already had the documents" is not consent. The builder takes one tenant per file, so the yes is one
# name on one line - and the file must not carry any other tenant's chunk. Every row still passes through
# eligible(): the tenant approved, an answerable row carries the quote it cites (a row without one would
# teach guessing), and a refusal row is there on purpose (every tenth chunk, 10.1), never by accident.
def eligible(msgs: list[dict]) -> tuple[bool, str]:
    """One row, one yes-or-no, one reason. Reasons get logged; rows do not."""
    tenant = msgs[0].get("tenant", TENANT)
    if tenant not in APPROVED_FOR_TRAINING:
        return False, "tenant has not approved training use"
    draft = json.loads(msgs[-1]["content"])
    if draft.get("answerable") and not any(c.get("quote") for c in draft.get("citations", [])):
        return False, "answer cites nothing - teaching this would teach guessing"
    if not draft.get("answerable") and draft.get("citations"):
        return False, "a refusal that cites - the model would learn to hedge with sources"
    return True, "ok"

try:
    CHAT_JSONL = gcs_text(f"gs://{DATASETS}/sft/documind_sft_v1.chat.jsonl")
    SOURCE = "frozen v1"
except Exception as e:
    print(f"no frozen file in the bucket ({type(e).__name__}); building a small one the same way (10.1's cell)")
    import make_trainset as mt
    chunks = mt.sample(mt.load_chunks(TENANT, PROJECT_ID), 30)
    rows = mt.ask_pairs(PROJECT_ID, chunks)
    golden = [json.loads(l) for l in open(f"{KIT}/deploy/evals/golden.jsonl", encoding="utf-8") if l.strip()]
    rows, _ = mt.exclude_golden(rows, golden)
    rows, _ = mt.redact(rows)
    CHAT_JSONL = "\n".join(json.dumps(mt.to_chat(r), ensure_ascii=False) for r in rows)
    SOURCE = "colab"
examples = [json.loads(l) for l in CHAT_JSONL.splitlines() if l.strip()]
verdicts = [eligible(e["messages"]) for e in examples]
kept = [e for e, (ok, _) in zip(examples, verdicts) if ok]
print(f"{SOURCE}: {len(examples)} rows, {len(kept)} eligible;", {why for ok, why in verdicts if not ok} or "no row dropped")
print("refusal rows kept on purpose:", sum(1 for e in kept if not json.loads(e["messages"][-1]["content"])["answerable"]))


## Cell 5: The file, read back


In [ ]:
# THE FILE, READ BACK THE WAY YOU WOULD READ ANY FILE. Three roles per row - the generator's SYSTEM, a user
# turn that is the lane's exact prompt shape ([Source 1] + the question), an assistant turn that is
# ModelDraft's JSON - because the habit taught is the habit the API serves: answer, cite by number, quote,
# say whether it was answerable. Not [doc:chunk_id] - the lane resolves numbers to chunks (resolve()), and a
# model that invents ids invents them convincingly.
from shared.documind_schemas import ModelDraft
with open("documind_sft.chat.jsonl", "w", encoding="utf-8") as f:
    for e in kept:
        f.write(json.dumps(e, ensure_ascii=False) + "\n")
lines = [json.loads(l) for l in open("documind_sft.chat.jsonl", encoding="utf-8") if l.strip()]
for i, ex in enumerate(lines, 1):
    roles = [m["role"] for m in ex["messages"]]
    assert roles == ["system", "user", "assistant"], (i, roles)
    ModelDraft.model_validate_json(ex["messages"][-1]["content"])          # the target parses, every row
print(f"{len(lines)} rows read back; every row has the three roles and a target ModelDraft parses")
print("\nuser      :", lines[0]["messages"][1]["content"][-200:].replace("\n", " | "))
print("assistant :", lines[0]["messages"][2]["content"][:200])


## Cell 6: Zero DLP findings, or it does not ship
Live, through `shared/pii.py`, and made to fail.


In [ ]:
from shared import pii

# ZERO DLP FINDINGS, OR IT DOES NOT SHIP. Retrieval can be filtered per tenant afterwards; WEIGHTS CANNOT. A
# model trained on a PAN can hand that PAN to a different customer, for the life of the adapter. This is the
# same shared/pii.py the ingest worker and the admin dashboard use - the ONE list - and it runs LIVE here on
# every user and assistant line, not a stand-in. The builder already applied it; a rule you re-check is a
# rule you know is working. Then the gate is made to fail, because a gate you have never seen fail is a
# gate you do not know works.
texts = [m["content"] for ex in lines for m in ex["messages"] if m["role"] != "system"]
findings = pii.inspect_many(texts)
hits = [(i, [f.get("info_type") for f in fs]) for i, fs in enumerate(findings) if fs]
print(f"DLP ({pii.LOCATION}) on {len(texts)} lines: {len(hits)} with findings")
assert not hits, hits[:3]

poisoned = texts + ["Contact Priya on 98765 43210, PAN ABCDE1234F, for the claim."]
bad = [fs for fs in pii.inspect_many(poisoned) if fs]
print("poisoned:", len(bad), "line(s) flagged:", sorted({f.get("info_type") for fs in bad for f in fs}))
assert bad, "the scan must catch a PAN - if it did not, nothing downstream is protected"
print("\nzero findings, or the file does not leave the bucket. There is no 'we will clean it later': later is after the weights exist.")


## Cell 7: Train it - needs a T4


In [ ]:
import torch
HAS_GPU = torch.cuda.is_available()

# TRAIN THE ADAPTER - on a T4 (Runtime -> Change runtime type -> T4 GPU). Three words, plainly:
#   quantisation  each number stored in fewer bits; 4-bit is a quarter of the memory for a loss you will not notice here
#   LoRA          a tiny extra layer trained beside the frozen weights; minutes, and a small file you can keep or discard
#   QLoRA         both: load 4-bit, train a LoRA on top - what makes a free T4 enough
# The pin is DATED: unsloth moves weekly and the version that ran is written into the manifest (Cell 10).
UNSLOTH = "unsloth>=2026.8.22"   # verified on a T4 on 2026-09-05; once a run is good, freeze the exact version pip reports
TRAIN_CODE = '''from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

# E2B is the small Gemma 4: fine-tunes on a FREE T4 in under twenty minutes. A bigger model is a slower way to find the same bugs.
model, tokenizer = FastLanguageModel.from_pretrained(model_name="unsloth/gemma-4-E2B-it", max_seq_length=2048, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0,          # rank 16 is plenty for a citation habit
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=42)
dataset = load_dataset("json", data_files="documind_sft.chat.jsonl", split="train")   # the messages column: system, user, assistant
trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=dataset,
                     args=TrainingArguments(per_device_train_batch_size=2, gradient_accumulation_steps=4, num_train_epochs=3,
                                            learning_rate=2e-4, fp16=True, logging_steps=5, output_dir="outputs", seed=42))
trainer.train()
model.save_pretrained("documind-slm-lora")                # the adapter, for the export step
tokenizer.save_pretrained("documind-slm-lora")
'''
open("train_documind_slm.py", "w", encoding="utf-8").write(TRAIN_CODE)
print("train_documind_slm.py written")

T4_USABLE_GB = 14.5
def vram_gb(params_b: float, bits: int, full_finetune: bool = False) -> float:
    """Rough, and rough is enough to decide. Full fine-tuning carries a gradient and two optimiser numbers per weight."""
    return params_b * bits / 8 * (4.0 if full_finetune else 1.8)
print(f"\n  {'model':13} {'bits':>5} {'method':6} {'needs':>9}   verdict")
for name, params_b, bits, full in (("Gemma 4 E2B", 2.0, 4, False), ("Gemma 4 E4B", 4.0, 4, False), ("Gemma 4 E4B", 4.0, 16, False), ("Gemma 4 E4B", 4.0, 16, True)):
    need = vram_gb(params_b, bits, full)
    print(f"  {name:13} {bits:>5} {'full' if full else 'LoRA':6} {need:>7.1f} GB   {'fits' if need < T4_USABLE_GB else 'DOES NOT FIT'}")

if HAS_GPU:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", UNSLOTH])
    subprocess.check_call([sys.executable, "train_documind_slm.py"])
    print("\ntrained: documind-slm-lora/")
else:
    print("\nno GPU on this runtime: switch to a T4, re-run from Setup, and this cell trains. Everything before it needed no GPU.")


## Cell 8: Export a GGUF under the name 11.4 expects


In [ ]:
# EXPORT A GGUF THAT OLLAMA CAN RUN - under the ONE name 11.4 builds from. GGUF is a file format for a model
# that runs without Python; q4_k_m keeps four bits per number with more precision where it matters. Unsloth
# writes unsloth.Q4_K_M.gguf; the Ollama image (services/slm/Dockerfile) copies exactly one object from
# exactly one place - gs://PROJECT-datasets/sft/documind-slm.gguf - so this is where the two lessons agree.
# The Hugging Face push is optional and private, and its token comes from Secret Manager (hf-token, secrets.tf)
# through gcloud - never typed into a cell, because a notebook gets shared and a pasted token with it.
EXPORT_CODE = '''import shutil, subprocess
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(model_name="documind-slm-lora", max_seq_length=2048, load_in_4bit=True)
model.save_pretrained_gguf("documind-slm", tokenizer, quantization_method="q4_k_m")
shutil.move("documind-slm/unsloth.Q4_K_M.gguf", "documind-slm.gguf")
'''
open("export_gguf.py", "w", encoding="utf-8").write(EXPORT_CODE)
if HAS_GPU and os.path.isdir("documind-slm-lora"):
    subprocess.check_call([sys.executable, "export_gguf.py"])
    gcs.bucket(DATASETS).blob("sft/documind-slm.gguf").upload_from_filename("documind-slm.gguf")
    print(f"documind-slm.gguf -> gs://{DATASETS}/sft/documind-slm.gguf  ({os.path.getsize('documind-slm.gguf') / 1e6:,.0f} MB)")
    PUSH_TO_HUB = False          # flip for a private repo: the token comes from Secret Manager, never from a cell
    if PUSH_TO_HUB:
        token = subprocess.run(["gcloud", "secrets", "versions", "access", "latest", "--secret=hf-token", "--project", PROJECT_ID],
                               capture_output=True, text=True, check=True).stdout.strip()
        subprocess.check_call([sys.executable, "-c", "from unsloth import FastLanguageModel; m, t = FastLanguageModel.from_pretrained('documind-slm-lora'); "
                               f"m.push_to_hub_gguf('your-org/documind-slm', t, quantization_method='q4_k_m', token='{token}', private=True)"])
else:
    print("export_gguf.py written; it runs after Cell 6 has trained on a T4")
print("\n11.4 picks the GGUF and its Modelfile up: make deploy-slm stages both from the bucket into an Ollama image on a Cloud Run L4. The gate for this lesson is that it loads and cites.")


## Cell 9: The Modelfile comes from the tokenizer


In [ ]:
sys.path.insert(0, f"{KIT}/deploy/services/slm")
from make_modelfile import build

# THE MODELFILE COMES FROM THE TOKENIZER. The GGUF carries the weights; the chat template and the stop tokens
# are the runtime's to get right, and a template typed from a blog post is the single most common way a
# working fine-tune looks broken - the model answers fluently and never stops. services/slm/make_modelfile.py
# renders the tokenizer's own template and takes the stop tokens from its special tokens; offline, the same
# build() shows the shape on a Gemma-style template.
if os.path.isdir("documind-slm-lora"):
    r = subprocess.run([sys.executable, f"{KIT}/deploy/services/slm/make_modelfile.py", "--model", "documind-slm-lora", "--gguf", "documind-slm.gguf"],
                       capture_output=True, text=True, check=True)
    open("Modelfile", "w", encoding="utf-8").write(r.stdout)
    print(r.stdout)
    # Beside the GGUF, under the name make deploy-slm fetches (Module 11): the template belongs to the weights, so the
    # two travel together and the image is built from both - never from a Modelfile typed into the repo.
    gcs.bucket(DATASETS).blob("sft/documind-slm.Modelfile").upload_from_filename("Modelfile")
    print(f"-> gs://{DATASETS}/sft/documind-slm.Modelfile (11.4's make deploy-slm stages it with the GGUF)")
else:
    print(build("<start_of_turn>user\n{{ .Prompt }}<end_of_turn>\n<start_of_turn>model\n", ["<eos>", "<end_of_turn>"], "documind-slm.gguf"))
    print("(offline shape; the real one is generated from documind-slm-lora's tokenizer after Cell 6)")
print("never stops -> stop tokens missing | wrong voice -> template wrong | right voice, wrong facts -> now it is your fine-tune")


## Cell 10: The other way - let Google do it


In [ ]:
from tune import TUNABLE, TUNABLE_DATE

# THE OTHER WAY - let Google do it (10.1). Same objective, opposite trade-offs; the base list is the kit's,
# dated, not retyped. DocuMind ships the OSS lane because its reason for tuning was RESIDENCY, and a managed
# endpoint in us-central1 does not solve that; the managed lane is the A/B that costs no effort.
print(f"{'':16} {'you run it (unsloth, this lesson)':34} {'Google runs it (Vertex SFT, 10.1)'}")
print("-" * 92)
for label, oss, managed in (("what you get",   "a GGUF file you own",            "an endpoint you call"),
                            ("where it runs",  "your Cloud Run, your region",     "Vertex, us-central1 (regional)"),
                            ("cost to train",  "free T4, about 20 minutes",       "per training token, billed"),
                            ("base model",     "any open weights",                f"{' or '.join(sorted(TUNABLE))} ONLY (as of {TUNABLE_DATE})"),
                            ("the A/B",        "compare_backends.py via 11.3",    "make candidate + make eval-live + make judge"),
                            ("can you keep it", "yes - it is a file",             "no - it lives in the project")):
    print(f"  {label:14} {oss:34} {managed}")


## Cell 11: Freeze it, and ship the manifest beside the GGUF


In [ ]:
import hashlib

# FREEZE IT, AND SHIP THE MANIFEST BESIDE THE GGUF. The dataset already has one (make trainset wrote it, with
# a sha and a date); the model gets its own: which dataset version, which base, which unsloth, which GGUF
# hash. Six months from now, when the model says something odd, "what was it trained on?" has an answer
# only if none of these moved. Never train on live traffic - the file is what it was on the day.
try:
    dataset_manifest = json.loads(gcs_text(f"gs://{DATASETS}/sft/documind_sft_v1.manifest.json"))
except Exception:
    dataset_manifest = {"version": SOURCE, "rows": len(lines), "note": "built in the notebook; make trainset freezes the real one"}
try:
    unsloth_version = subprocess.run([sys.executable, "-m", "pip", "show", "unsloth"], capture_output=True, text=True).stdout.split("Version:")[1].split()[0]
except IndexError:
    unsloth_version = "not installed on this runtime"
manifest = {
    "built_at": datetime.date.today().isoformat(),
    "dataset": {k: dataset_manifest.get(k) for k in ("version", "built_at", "rows", "refusals", "dropped_pii", "corpus_manifest_sha256")},
    "dataset_rows_used": len(lines), "tenants_approved": sorted(APPROVED_FOR_TRAINING),
    "dlp_findings": 0, "base_model": "unsloth/gemma-4-E2B-it", "method": "QLoRA r=16, 3 epochs, T4", "unsloth": unsloth_version,
    "gguf": {"name": "documind-slm.gguf", "quantization": "q4_k_m",
             "sha256": hashlib.sha256(open("documind-slm.gguf", "rb").read()).hexdigest()[:16] if os.path.isfile("documind-slm.gguf") else None},
}
print(json.dumps(manifest, indent=1))
if manifest["gguf"]["sha256"]:
    gcs.bucket(DATASETS).blob("sft/documind-slm.manifest.json").upload_from_string(json.dumps(manifest, indent=1))
    print(f"\n-> gs://{DATASETS}/sft/documind-slm.manifest.json, beside the GGUF 11.4 builds from")


## Where this goes
- **10.6** teaches the same model to reason, with a reward written on the contract.
- **11.4** bakes the GGUF into an Ollama image on Cloud Run with an L4 and prices it against gemini-3.6-flash; **compare_backends.py** is the pairwise.

## ✅ Lesson 10.5 complete
- ✅ The decision gate; the data's provenance read off a real usage row
- ✅ Consent as the tenant's; the same frozen dataset, in its chat format, read back and parsed
- ✅ DLP live through the one list, with a poisoned row that fails
- ✅ QLoRA on a T4 behind a GPU check; the GGUF and its manifest in the datasets bucket
- ✅ The Modelfile from the tokenizer; the managed alternative with the kit's dated base list
